In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
pyspark.__version__

'3.5.5'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/06 21:51:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# !wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [3]:
# !wc -l fhvhv_tripdata_2021-01.csv

wc: fhvhv_tripdata_2021-01.csv: No such file or directory


In [5]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [6]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [6]:
# !gzip -dc fhvhv_tripdata_2021-01.csv.gz | head -n 1001 > head.csv


gzip: stdout: Broken pipe


In [7]:
import pandas as pd


In [8]:
df_pandas = pd.read_csv('head.csv')

In [9]:
df_pandas.dtypes

hvfhs_license_num        object
dispatching_base_num     object
pickup_datetime          object
dropoff_datetime         object
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [10]:
spark.createDataFrame(df_pandas)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: string, dropoff_datetime: string, PULocationID: bigint, DOLocationID: bigint, SR_Flag: double]

In [11]:
from pyspark.sql import types

In [12]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [13]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv.gz')

In [14]:
df = df.repartition(24)

In [16]:
df.write.parquet('fhvhv/2021/01/', mode="overwrite")

In [17]:
df = spark.read.parquet('fhvhv/2021/01/')

In [18]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [19]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|
|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|
|2021-01-02 00:34:43|2021-01-02 00:45:38|          63|          77|
|2021-01-02 16:20:11|2021-01-02 16:56:36|          63|         244|
|2021-01-24 16:00:53|2021-01-24 16:07:40|         210|         165|
|2021-01-16 19:35:17|2021-01-16 19:50:20|         113|         143|
|2021-01-01 11:15:17|2021-01-01 11:24:55|         231|         148|
|2021-01-19 12:05:32|2021-01-19 12:33:46|         228|         210|
|2021-01-17 13:54:52|2021-01-17 14:07:03|          39|          61|
|2021-01-30 18:03:33|2021-01-30 18:23:17|          42|         250|
|2021-01-16 12:36:55|2021-01-16 13:03:23|         131|         265|
|2021-01-30 23:07:14|2021-01-30 23:27:34|       

In [20]:
from pyspark.sql import functions as F

def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [21]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()


+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-10|  2021-01-10|          97|          25|
|  e/9ce| 2021-01-08|  2021-01-08|         138|         265|
|  e/b3c| 2021-01-01|  2021-01-01|          50|         163|
|  e/9ce| 2021-01-15|  2021-01-15|         163|          79|
|  e/9ce| 2021-01-12|  2021-01-12|          47|          74|
|  s/acd| 2021-01-05|  2021-01-05|          48|          95|
|  e/b38| 2021-01-02|  2021-01-02|          63|          77|
|  e/9ce| 2021-01-06|  2021-01-06|         238|          41|
|  e/acc| 2021-01-02|  2021-01-02|          63|         244|
|  e/acc| 2021-01-24|  2021-01-24|         210|         165|
|  e/b35| 2021-01-16|  2021-01-16|         113|         143|
|  e/9ce| 2021-01-28|  2021-01-28|          91|          89|
|  e/acc| 2021-01-01|  2021-01-01|         231|         148|
|  e/b33| 2021-01-19|  2